# Pessoa DRR 17 IE-ST

Notebook de operacao para o snapshot `DIM_PESSOA` x `DIM_DRR` usado no cruzamento com a extracao anterior.

Fluxo:
1. carregar o helper local;
2. revisar os parametros e a SQL;
3. rodar dry-run;
4. executar o snapshot;
5. inspecionar `chunks`, `consolidado`, `top_n` e `manifest`.


In [ ]:
from pathlib import Path
import sys

from IPython.display import display


def find_root(start: Path | None = None) -> Path:
    start = start or Path.cwd()
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "freeform_snapshot_export.py").exists():
            return candidate
    raise RuntimeError("Nao foi possivel localizar a raiz do projeto.")


ROOT = find_root()
WORKSPACE_ROOT = ROOT / "tb_cadastro"
if str(WORKSPACE_ROOT) not in sys.path:
    sys.path.insert(0, str(WORKSPACE_ROOT))

import download_tb_cadastro as snapshot

ROOT, WORKSPACE_ROOT


In [ ]:
TOP_N = 20
PAGE_SIZE = 5000
MAX_ATTEMPTS = 2
SLEEP_BETWEEN_ATTEMPTS = 5
OUTPUT_DIR = WORKSPACE_ROOT / "outputs"
OUTPUT_PATH = OUTPUT_DIR / snapshot.DEFAULT_OUTPUT_FILENAME
DRY_RUN = True
RUN_EXPORT = False

{
    "TOP_N": TOP_N,
    "PAGE_SIZE": PAGE_SIZE,
    "MAX_ATTEMPTS": MAX_ATTEMPTS,
    "OUTPUT_PATH": str(OUTPUT_PATH),
}


In [ ]:
plan = snapshot.dry_run_plan(
    top_n=TOP_N,
    page_size=PAGE_SIZE,
    output_dir=OUTPUT_DIR,
    output_path=OUTPUT_PATH,
)
plan


In [ ]:
run = None
if RUN_EXPORT:
    run = snapshot.run_tb_cadastro_export(
        top_n=TOP_N,
        page_size=PAGE_SIZE,
        output_dir=OUTPUT_DIR,
        output_path=OUTPUT_PATH,
        max_attempts=MAX_ATTEMPTS,
        sleep_between_attempts=SLEEP_BETWEEN_ATTEMPTS,
        verbose=True,
    )
    run.summary
else:
    print("Set RUN_EXPORT = True to execute the snapshot.")


In [ ]:
if run is not None:
    display(run.result.chunks.head())
    display(run.result.consolidated.head())
    display(run.result.top_n.head())
    display(run.result.manifest)
